# manual-chain-forward-and-back — faded example 3: Manual Backward Through a 3-Step abs → square → exp Chain

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `manual-chain-forward-and-back`. Running the beacon reports progress on the `Backprop: manual chain forward-and-back` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: manual chain forward-and-back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`manual-chain-forward-and-back`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "manual-chain-forward-and-back"
DD_SUBTOPIC = "Backprop: manual chain forward-and-back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A three-operation chain requires three backward functions applied in reverse order. Given `b = |a|`, `c = b²`, `d = exp(c)`, the backward sequence is exp_back → square_back → abs_back. Each backward function uses either the cached output or the cached input of its forward operation — the precise argument passed is determined by which form the derivative takes.

## Faded exercise 3

Complete the function below. The forward pass and the first two backward steps are done. You must fill in the single blanked line that computes `dL/db` using `square_back`.

`square_back(grad_out, out, x)` returns `grad_out * 2 * x`, where `x` is the input to the squaring step (i.e., `b`).

**Fill in:** Apply square_back to compute dL/db from dL/dc, using input b.

In [ ]:
import torch as t

def exp_back(grad_out, out, x):
    return grad_out * out

def square_back(grad_out, out, x):
    return grad_out * 2.0 * x

def abs_back(grad_out, out, x):
    return grad_out * t.sign(x)

def manual_abs_square_exp_chain(a, dL_dd):
    # Forward: a -> b -> c -> d
    b = t.abs(a)
    c = b ** 2
    d = t.exp(c)
    # Backward (reverse order)
    dL_dc = exp_back(dL_dd, d, c)
    dL_db = square_back(dL_dc, c, b)
    dL_da = abs_back(dL_db, b, a)
    return b, c, d, dL_dc, dL_db, dL_da

t.manual_seed(66)
a_val = t.tensor([-2.0, 1.0, -0.5, 3.0])
dL_dd_val = t.ones(4)
b, c, d, dL_dc, dL_db, dL_da = manual_abs_square_exp_chain(a_val, dL_dd_val)
print(f"dL/da (manual): {dL_da}")


import torch as t

def _test():
    t.manual_seed(66)
    a_val = t.tensor([-2.0, 1.0, -0.5, 3.0])
    dL_dd_val = t.ones(4)

    b, c, d, dL_dc, dL_db, dL_da = manual_abs_square_exp_chain(a_val, dL_dd_val)

    # Verify against autograd (skip a=0 edge case)
    a_ag = a_val.clone().requires_grad_(True)
    d_ag = t.exp((t.abs(a_ag)) ** 2)
    loss = (d_ag * dL_dd_val).sum()
    loss.backward()

    # At non-zero points both agree; a=0 is an edge case excluded from test data
    assert t.allclose(dL_da, a_ag.grad, atol=1e-4), \
        f"dL/da mismatch: manual={dL_da}, autograd={a_ag.grad}"
    # Also check the intermediate dL/db
    expected_dL_db = dL_dc * 2.0 * b
    assert t.allclose(dL_db, expected_dL_db, atol=1e-5)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def exp_back(grad_out, out, x):
    return grad_out * out

def square_back(grad_out, out, x):
    return grad_out * 2.0 * x

def abs_back(grad_out, out, x):
    return grad_out * t.sign(x)

def manual_abs_square_exp_chain(a, dL_dd):
    # Forward: a -> b -> c -> d
    b = t.abs(a)
    c = b ** 2
    d = t.exp(c)
    # Backward (reverse order)
    dL_dc = exp_back(dL_dd, d, c)
    dL_db = square_back(dL_dc, c, b)
    dL_da = abs_back(dL_db, b, a)
    return b, c, d, dL_dc, dL_db, dL_da

t.manual_seed(66)
a_val = t.tensor([-2.0, 1.0, -0.5, 3.0])
dL_dd_val = t.ones(4)
b, c, d, dL_dc, dL_db, dL_da = manual_abs_square_exp_chain(a_val, dL_dd_val)
print(f"dL/da (manual): {dL_da}")
```
</details>